# PISM-Cloud

PISM-Cloud is a custom AWS deployment of the Alaska Satellite Facilities' serverless, batch processing system, HyP3. PISM-Cloud allows you to run PISM simulations of Alaska glaciers by just providing a RGI glacier complex ID, PISM configuration files, and a run template.

## Connect to PISM-Cloud

In order to equitably distribute PISM-Cloud resources, users must connect to the API with NASA Earthdata Login credentials and be assigned simulation credits.

[Earthdata Login](https://urs.earthdata.nasa.gov/ "https://urs.earthdata.nasa.gov/" )
(EDL) is the authentication method used across NASA's [Earth Observation System Data Information System (EOSDIS)](https://www.earthdata.nasa.gov/about/esdis/eosdis "www.earthdata.nasa.gov/about/esdis/eosdis" ). These credentials allow you to access to any of the Earth Science data products served by EOSDIS, and for our purposes, PISM-Cloud.

There is no cost to [register for EDL credentials](https://urs.earthdata.nasa.gov/users/new "https://urs.earthdata.nasa.gov/users/new" ), and the process is quick and easy. When creating your profile, make sure to select an item in the **Study Area** field, as you may encounter access errors if that field is left blank.

You can connect to PISM-Cloud using the HyP3 SDK like:

In [ ]:
import fsspec
import s3fs
import geopandas as gpd
from pathlib import Path
from copy import deepcopy
import hyp3_sdk as sdk

pism_cloud = sdk.HyP3('https://pism-cloud-test.asf.alaska.edu', prompt='password')

and check the number of credits you have with:

In [ ]:
pism_cloud.check_credits()

We choose the Wrangell Glacier Complex

In [ ]:
rgi_id = 'RGI2000-v7.0-C-01-04374'

In [ ]:
name = "summer_school"
bucket = "pism-cloud-data"
project = "test_inverse_fridays"
bucket_prefix = f"{{name}}/{project}/{{job_id}}"

## Preparing simulations

Simulation "jobs" are requested in two steps. First, we need to prepare all the necessary input data for the glacier complexes we want to simulate by submitting a PISM_TERRA_RUN_FORWARD (PISM_TERRA_RUN_INVERSE) job using the template below. These jobs will create the simulation grids; project the input DEM, surface elevations, velocities, etc., onto the grids; and generate executable scripts that will run the simulation.

In [ ]:
import boto3
import ipywidgets as widgets
from IPython.display import display


pism_config: str | None = None
pism_template: str | None = None
pism_uq: str | None = "none"

_s3 = boto3.client("s3")


def _make_uploader(kind: str, var_name: str, accept: str = "") -> widgets.VBox:
    """
    Build a FileUpload widget that ships the dropped file to S3.

    Parameters
    ----------
    kind : str
        Sub-directory under ``s3://{bucket}/glacier/{name}/`` (e.g. ``"config"``).
    var_name : str
        Name of the notebook variable to bind to the resulting ``s3://…`` URI.
    accept : str, optional
        Comma-separated list of accepted file extensions for the picker
        (e.g. ``".toml,.yaml"``). Empty string accepts any file.
    """
    header = widgets.HTML(value=f"<b>Upload {kind} file</b>")
    picker = widgets.FileUpload(accept=accept, multiple=False, description=f"Browse {kind}…")
    status = widgets.HTML(value="<i>no file uploaded yet</i>")

    def _on_change(_change):
        if not picker.value:
            return
        # ipywidgets ≥ 8 returns a tuple of dicts; older versions return a dict.
        item = picker.value[0] if isinstance(picker.value, (list, tuple)) else next(iter(picker.value.values()))
        filename = item.get("name") or item["metadata"]["name"]
        raw = item["content"] if isinstance(item.get("content"), bytes) else bytes(item["content"])
        key = f"glacier/{name}/{kind}/{filename}"
        _s3.put_object(Bucket=bucket, Key=key, Body=raw)
        uri = f"s3://{bucket}/{key}"
        globals()[var_name] = uri
        status.value = f"✅ uploaded → <code>{uri}</code>"

    picker.observe(_on_change, names="value")
    return widgets.VBox([header, picker, status])

### Config file

Select the `config` file you want to upload. The `config` file is *TOML* file containing a list of options needed to run PISM.

In [ ]:
display(
    widgets.VBox(
        [
            _make_uploader("config", "pism_config", accept=".toml"),
        ]
    )
)


### Template file

Select the `template` file you want to upload. The `template` file is a *Jinja2* file that controls system architecture specific execution. Choose either `ec2.j2` for forward simulations or `ec2-inverse.j2` for forward-inverse simulations.

In [ ]:
display(
    widgets.VBox(
        [
            _make_uploader("template", "pism_template", accept=".j2,.jinja,.jinja2"),
        ]
    )
)


### UQ file

Select the  *optional* `UQ` file you want to upload. The `UQ` file is a *TOML* file that, if given, controls the uncertainty qunatification.

In [ ]:
display(
    widgets.VBox(
        [
            _make_uploader("uq", "pism_uq", accept=".toml"),
        ]
    )
)


Now we need to set up the PISM staging job, either for a `forward` or mixed `forward/inverse` run.

In [ ]:
foward_job =     {
    "job_type": "PISM_TERRA_RUN_FORWARD",
    "name": name,
    "bucket": bucket,
    "bucket_prefix": bucket_prefix,
    "job_parameters": {
        "rgi_id": rgi_id,
        "pism_config": pism_config,
        "run_template": pism_template,
        "uq_config": pism_uq,
        "ntasks": 32,
    }
}

inverse_job =     {
    "job_type": "PISM_TERRA_RUN_INVERSE",
    "name": name,
    "bucket": bucket,
    "bucket_prefix": bucket_prefix,
    "job_parameters": {
        "rgi_id": rgi_id,
        "pism_config": pism_config,
        "run_template": pism_template,
        "uq_config": pism_uq,
        "ntasks": 32,
    }
}

print(name)
print(bucket)
print(bucket_prefix)

pism_job = inverse_job

Then we'll loop through all the RGIs we want to simulate, set the `rgi_id` job parameter, and give the job an appropriate name so we can easily find it later.

In [ ]:
jobs = pism_cloud.submit_prepared_jobs(pism_job)
print(jobs)

We can use the `watch` method to periodically check on our jobs and tell us when they have finished.

In [ ]:
jobs = pism_cloud.watch(jobs)

## Execute the PISM-Cloud simulation

The prepare jobs will have created a set of simulations run scripts, which we need to find and then tell PISM-Cloud to execute. All job outputs are uploaded to a "content" bucket under a `{name}/{project}/{job_id}` prefix (folder). This function will look in the prefix for all run scripts and return a list of their paths.

In [ ]:
def get_run_scripts(job: sdk.Job) ->  list[str]:
    fs = s3fs.S3FileSystem(anon=True)
    files = fs.ls(f"{bucket}/{job.name}/{project}/{job.job_id}/{rgi_id}/run_scripts/")
    return [f's3://{file}' for file in files]

And now, using the execute template, we can run the simulation.

In [ ]:
EXECUTE_TEMPLATE = {
    "name": name,
    "bucket": bucket,
    "bucket_prefix": bucket_prefix,
    "job_type": "PISM_TERRA_EXECUTE",
    "job_parameters": {
    }
}

prepared_jobs = []
for job in jobs:
    run_scripts = get_run_scripts(job)
    for script in run_scripts:
        job_dict = deepcopy(EXECUTE_TEMPLATE)
        job_dict['name'] = job.name
        job_dict['job_parameters']['run_script'] = script
        prepared_jobs.append(job_dict)

In [ ]:
jobs = pism_cloud.submit_prepared_jobs(prepared_jobs)
print(jobs)

In [ ]:
jobs = pism_cloud.watch(jobs)

In [ ]:
jobs